In [1]:


#%% 
!pip install --no-index --find-links=/kaggle/input/tf-efficientnet-whl-files /kaggle/input/tf-efficientnet-whl-files/efficientnet-1.1.1-py3-none-any.whl
PLATFORM = 'kaggle' # local kaggle
NEEDTRAIN = False
LOAD_MODELS_FROM = 'models20240118'
if PLATFORM == 'local':
    LOAD_MODELS_FROM = f'./input/{LOAD_MODELS_FROM}'
elif PLATFORM == 'kaggle':
    LOAD_MODELS_FROM = f'/kaggle/input/{LOAD_MODELS_FROM}'

EEG_LENGTH = 20.48 # s
SFREQ = 200

HIGH = 128 # 128
LENGTH = 256 # 256

READ_SPEC_FILES = True
READ_EEG_FILES = True

filter_range = [1, 40]

BRAIN = {'LL': ['Fp1-F7', 'F7-T3', 'T3-T5', 'T5-O1'],
         'RL': ['Fp2-F8', 'F8-T4', 'T4-T6', 'T6-O2'],
         'LP': ['Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1'],
         'RP': ['Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2']}

import os
os.environ["CUDA_VISIBLE_DEVICES"]="0, 1"
import tensorflow as tf
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from tensorflow.python.framework.ops import reset_default_graph
print('TensorFlow version =',tf.__version__)

# USE MULTIPLE GPUS
gpus = tf.config.list_physical_devices('GPU')
if len(gpus)<=1: 
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    print(f'Using {len(gpus)} GPU')
else: 
    strategy = tf.distribute.MirroredStrategy()
    print(f'Using {len(gpus)} GPUs')

VER = 1

# IF THIS EQUALS NONE, THEN WE TRAIN NEW MODELS
# IF THIS EQUALS DISK PATH, THEN WE LOAD PREVIOUSLY TRAINED MODELS
# LOAD_MODELS_FROM = '/kaggle/input/brain-efficientnetb2-models-v1/'



#%% USE MIXED PRECISION
MIX = True
if MIX:
    tf.config.optimizer.set_experimental_options({"auto_mixed_precision": True})
    print('Mixed precision enabled')
else:
    print('Using full precision')
    

#%% Load Train Data
if PLATFORM == 'local':
    df = pd.read_csv('./input/hms-harmful-brain-activity-classification/train.csv')
elif PLATFORM == 'kaggle':
    df = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/train.csv')
TARGETS = df.columns[-6:]
print('Train shape:', df.shape )
print('Targets', list(TARGETS))
df.head()


#%% Create Non-Overlapping EEG Id Train Data
train = df.groupby('eeg_id')[['spectrogram_id', 'spectrogram_label_offset_seconds',
                              'eeg_label_offset_seconds']].agg({'spectrogram_id':'first',
                                                                'spectrogram_label_offset_seconds':'min',
                                                                'eeg_label_offset_seconds': 'median'})
train.columns = ['spec_id', 'min', 'eeg_median']

tmp = df.groupby('eeg_id')[['spectrogram_id','spectrogram_label_offset_seconds']].agg(
    {'spectrogram_label_offset_seconds':'max'})

train['max'] = tmp

tmp = df.groupby('eeg_id')[['patient_id']].agg('first')
train['patient_id'] = tmp

tmp = df.groupby('eeg_id')[TARGETS].agg('sum')
for t in TARGETS:
    train[t] = tmp[t].values
    
y_data = train[TARGETS].values
y_data = y_data / y_data.sum(axis=1,keepdims=True)
train[TARGETS] = y_data

tmp = df.groupby('eeg_id')[['expert_consensus']].agg('first')
train['target'] = tmp

train = train.reset_index()
print('Train non-overlapp eeg_id shape:', train.shape )
train.head()


#%% Read Train Spectrograms
if NEEDTRAIN:
    
    
    # READ ALL SPECTROGRAMS
    if PLATFORM == 'local':
        PATH = './input/hms-harmful-brain-activity-classification/train_spectrograms/'
    elif PLATFORM == 'kaggle':
        PATH = '/kaggle/input/hms-harmful-brain-activity-classification/train_spectrograms/'
    files = os.listdir(PATH)
    print(f'There are {len(files)} spectrogram parquets')
    
    if READ_SPEC_FILES:    
        spectrograms = {}
        for i,f in enumerate(files):
            if i%100==0: print(i,', ',end='')
            tmp = pd.read_parquet(f'{PATH}{f}')
            name = int(f.split('.')[0])
            spectrograms[name] = tmp.iloc[:,1:].values
        if not os.path.exists('./input/brain-spectrograms'):
            os.makedirs('./input/brain-spectrograms')
        np.save('./input/brain-spectrograms/specs.npy', spectrograms, allow_pickle=True)
    else:
        if PLATFORM == 'local':
            spectrograms = np.load('./input/brain-spectrograms/specs.npy', allow_pickle=True).item()
        elif PLATFORM == 'kaggle':
            spectrograms = np.load('/kaggle/input/brain-spectrograms/specs.npy', allow_pickle=True).item()
        
        
#%% Read Train EEGs
if NEEDTRAIN:
    
    from scipy import signal
    
    # READ ALL SPECTROGRAMS
    if PLATFORM == 'local':
        PATH = './input/hms-harmful-brain-activity-classification/train_eegs/'
    elif PLATFORM == 'kaggle':
        PATH = '/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/'
    files = os.listdir(PATH)
    print(f'There are {len(files)} eeg parquets')
    
    if READ_EEG_FILES:    
        eegs = {}
        b, a = signal.butter(3, np.float32(filter_range)*2/SFREQ, 'bandpass')
        for i,f in enumerate(files):
            if i%100==0: print(i,', ',end='')
            raw_eeg = pd.read_parquet(f'{PATH}{f}')
            name = int(f.split('.')[0])
            
            if len(train[train.eeg_id == name]) > 0:
                time_temp = train[train.eeg_id == name].eeg_median.iloc[-1]
                time_start =  round(time_temp + (50 - EEG_LENGTH) / 2 * 200) 
                time_stop =  round(time_temp + (50 + EEG_LENGTH) / 2 * 200)
        
                eeg_default = raw_eeg.loc[time_start: (time_stop - 1), :].reset_index(drop=True)
                
                list_eeg = list()
                for region in BRAIN.keys():
        
                    eeg = np.zeros((len(BRAIN[region]), eeg_default.shape[0]), dtype=np.float32)
                    for chan_i, chan in enumerate(BRAIN[region]):
                        eeg[chan_i, :] = (eeg_default.loc[:, chan.split('-')[0]] - eeg_default.loc[:, chan.split('-')[1]]).values
                    
                    eeg[np.isnan(eeg)] = 0
                    
                    if 200 != SFREQ:
                        eeg = signal.resample_poly(eeg, SFREQ, 200, axis=1)
        
                    eeg = signal.filtfilt(b, a, eeg, axis=1)
                    
                    
                    
                    list_eeg.append(np.reshape(eeg, (eeg.shape[0], eeg.shape[1], 1)))
                    
                list_eeg = np.concatenate(list_eeg, 2)
                
                eegs[name] = list_eeg
            
        if not os.path.exists('./input/brain-eegs'):
            os.makedirs('./input/brain-eegs')
        np.save('./input/brain-eegs/eegs.npy', eegs, allow_pickle=True)
    else:
        if PLATFORM == 'local':
            eegs = np.load('./input/brain-eegs/eegs.npy', allow_pickle=True).item()
        elif PLATFORM == 'kaggle':
            eegs = np.load('/kaggle/input/brain-eegs/eegs.npy', allow_pickle=True).item()


#%% Train DataLoader

import albumentations as albu
TARS = {'Seizure':0, 'LPD':1, 'GPD':2, 'LRDA':3, 'GRDA':4, 'Other':5}
TARS2 = {x:y for y,x in TARS.items()}

class DataGenerator(tf.keras.utils.Sequence):
    'Generates data for Keras'
    def __init__(self, data, batch_size=32, shuffle=False, augment=False, mode='train',
                 specs = None, eegs = None): 

        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = False
        self.mode = mode
        self.specs = specs
        self.eegs = eegs
        self.on_epoch_end()
        
    def __len__(self):
        'Denotes the number of batches per epoch'
        ct = int( np.ceil( len(self.data) / self.batch_size ) )
        return ct

    def __getitem__(self, index):
        'Generate one batch of data'
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        X, X_eeg, y = self.__data_generation(indexes)
        if self.augment: X = self.__augment_batch(X) 
        return [X, X_eeg], y

    def on_epoch_end(self):
        'Updates indexes after each epoch'
        self.indexes = np.arange( len(self.data) )
        if self.shuffle: np.random.shuffle(self.indexes)
                        
    def __data_generation(self, indexes):
        'Generates data containing batch_size samples' 
        
        X = np.zeros((len(indexes),HIGH,LENGTH,4),dtype='float32')
        X_eeg = np.zeros((len(indexes),4,round(EEG_LENGTH * SFREQ),4),dtype='float32')
        # y = np.zeros((len(indexes)),dtype='int32')
        y = np.zeros((len(indexes), 6),dtype='float32')
        img = np.ones((HIGH,LENGTH),dtype='float32')
        img_eeg = np.ones((4,round(EEG_LENGTH * SFREQ)),dtype='float32')
        
        for j,i in enumerate(indexes):
            row = self.data.iloc[i]
            if self.mode=='test': 
                r = 0
            elif self.mode=='valid': 
                r = int( (row['min'] + row['max'])//4 )
            else:
                # RANDOM CROPS FOR TRAIN
                r = np.random.randint(row['min'], row['max']+1)//2  

            for k in range(4):
                # EXTRACT 300 ROWS OF SPECTROGRAM
                img = self.specs[row.spec_id][r:r+300,k*100:(k+1)*100].T
                img_eeg = self.eegs[row.eeg_id][:, :, k]
                
                # LOG TRANSFORM SPECTROGRAM
                img = np.clip(img,np.exp(-4),np.exp(8))
                img = np.log(img)
                
                img = np.nan_to_num(img, nan=0.0)
                
                # CROP TO 256 TIME STEPS
                if HIGH <= 100:
                    img = np.resize(img[:,round((600 / 2 - LENGTH) / 2):(round((600 / 2 - LENGTH) / 2) + LENGTH)], (HIGH, LENGTH))
                else:
                    img = img[:,round((600 / 2 - LENGTH) / 2):(round((600 / 2 - LENGTH) / 2) + LENGTH)]
                
                # STANDARDIZE PER IMAGE
                # ep = 1e-6
                # m = np.nanmean(img.flatten())
                # s = np.nanstd(img.flatten())
                # img = (img-m)/(s+ep)
                # img = np.nan_to_num(img, nan=0.0)
                
                if HIGH <= 100:
                    X[j,:,:,k] = img
                else:
                    X[j,round((HIGH - 100) / 2):-round((HIGH - 100) / 2),:,k] = img
                
                X_eeg[j, :, :, k] = img_eeg
            
            X[j, :, :, :] = (X[j, :, :, :] - np.mean(X[j, :, :, :])) / (np.std(X[j, :, :, :]) + 1e-6)
            X_eeg[j, :, :, :] = (X_eeg[j, :, :, :] - np.mean(X_eeg[j, :, :, :])) / (np.std(X_eeg[j, :, :, :]) + 1e-6)
                    
            if self.mode!='test':
                # y[j] = TARS[row.target]
                y[j] = row[TARGETS].values
        
        return X,X_eeg,y
    
    def __random_transform(self, img):
        composition = albu.Compose([
            albu.HorizontalFlip(p=0.5),
            albu.CoarseDropout(max_holes=8,max_height=32,max_width=32,fill_value=0,p=0.5),
        ])
        return composition(image=img)['image']
            
    def __augment_batch(self, img_batch):
        for i in range(img_batch.shape[0]):
            img_batch[i, ] = self.__random_transform(img_batch[i, ])
        return img_batch


#%% Train Scheduler
if NEEDTRAIN:
    import math
    LR_START = 1e-6
    LR_MAX = 1e-3
    LR_MIN = 1e-6
    LR_RAMPUP_EPOCHS = 0
    LR_SUSTAIN_EPOCHS = 0
    EPOCHS2 = 10
    
    def lrfn(epoch):
        if epoch < LR_RAMPUP_EPOCHS:
            lr = (LR_MAX - LR_START) / LR_RAMPUP_EPOCHS * epoch + LR_START
        elif epoch < LR_RAMPUP_EPOCHS + LR_SUSTAIN_EPOCHS:
            lr = LR_MAX
        else:
            decay_total_epochs = EPOCHS2 - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS - 1
            decay_epoch_index = epoch - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS
            phase = math.pi * decay_epoch_index / decay_total_epochs
            cosine_decay = 0.5 * (1 + math.cos(phase))
            lr = (LR_MAX - LR_MIN) * cosine_decay + LR_MIN
        return lr
    
    rng = [i for i in range(EPOCHS2)]
    lr_y = [lrfn(x) for x in rng]
    plt.figure(figsize=(10, 4))
    plt.plot(rng, lr_y, '-o')
    plt.xlabel('epoch',size=14); plt.ylabel('learning rate',size=14)
    plt.title('Cosine Training Schedule',size=16); plt.show()
    
    LR2 = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose = True)
    
    
    LR_START = 1e-4
    LR_MAX = 1e-3
    LR_RAMPUP_EPOCHS = 0
    LR_SUSTAIN_EPOCHS = 0
    LR_STEP_DECAY = 0.1
    EVERY = 2
    EPOCHS = 5
    
    def lrfn(epoch):
        if epoch < LR_RAMPUP_EPOCHS:
            lr = (LR_MAX - LR_START) / LR_RAMPUP_EPOCHS * epoch + LR_START
        elif epoch < LR_RAMPUP_EPOCHS + LR_SUSTAIN_EPOCHS:
            lr = LR_MAX
        else:
            lr = LR_MAX * LR_STEP_DECAY**((epoch - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS)//EVERY)
        return lr
    
    rng = [i for i in range(EPOCHS)]
    y = [lrfn(x) for x in rng]
    plt.figure(figsize=(10, 4))
    plt.plot(rng, y, 'o-'); 
    plt.xlabel('epoch',size=14); plt.ylabel('learning rate',size=14)
    plt.title('Step Training Schedule',size=16); plt.show()
    
    LR = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose = True)

    # EPOCHS = 6
    # def lrfn(epoch):
    #     epoch = epoch + 1
    #     if (epoch == 1) or (epoch == 2):
    #         lr = 1e-3
    #     elif (epoch >= 3) and (epoch <= 6):
    #         lr = 1e-4
    #     elif (epoch >= 6) and (epoch <= 10):
    #        lr = 1e-5
    
    #     return lr
    
    # rng = [i for i in range(EPOCHS)]
    # y = [lrfn(x) for x in rng]
    # plt.figure(figsize=(10, 4))
    # plt.plot(rng, y, 'o-'); 
    # plt.xlabel('epoch',size=14); plt.ylabel('learning rate',size=14)
    # plt.title('Step Training Schedule',size=16); plt.show()
    
    # LR = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose = True)


#%% Build EfficientNetB2 Model
import efficientnet.tfkeras as efn
# from input.swintransformer.model import SwinTransformer
def wave_block(x, filters, kernel_size, n):
    dilation_rates = [2**i for i in range(n)]
    x = tf.keras.layers.Conv1D(filters = filters,
               kernel_size = 1,
               padding = 'same')(x)
    res_x = x
    for dilation_rate in dilation_rates:
        tanh_out = tf.keras.layers.Conv1D(filters = filters,
                          kernel_size = kernel_size,
                          padding = 'same', 
                          activation = 'tanh', 
                          dilation_rate = dilation_rate)(x)
        sigm_out = tf.keras.layers.Conv1D(filters = filters,
                          kernel_size = kernel_size,
                          padding = 'same',
                          activation = 'sigmoid', 
                          dilation_rate = dilation_rate)(x)
        x = tf.keras.layers.Multiply()([tanh_out, sigm_out])
        x = tf.keras.layers.Conv1D(filters = filters,
                   kernel_size = 1,
                   padding = 'same')(x)
        res_x = tf.keras.layers.Add()([res_x, x])
    return res_x


def build_model():
    
    inp = tf.keras.Input(shape=(HIGH,LENGTH,4))
    inp_eeg = tf.keras.Input(shape=(4,round(EEG_LENGTH * SFREQ),4))
    
    base_model = efn.EfficientNetB2(include_top=False, weights=None, input_shape=None)
    if PLATFORM == 'local':
        base_model.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnet-b2_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
    if PLATFORM == 'kaggle':
        base_model.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnet-b2_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')

    
    ## spectrograms 
    # RESHAPE INPUT 128x256x4 => 512x256x3 MONOTONE IMAGE
    x0 = inp[:,:,:,:1]
    x1 = inp[:,:,:,1:2]
    x2 = inp[:,:,:,2:3]
    x3 = inp[:,:,:,3:4]
    x01 = tf.keras.layers.Concatenate(axis=1)([x0,x1,x2,x3])

    x = tf.keras.layers.Concatenate(axis=3)([x01,x01,x01])
    
    # OUTPUT
    x = base_model(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    
    
    
    base_model_eeg = efn.EfficientNetB1(include_top=False, weights=None, input_shape=None)
    if PLATFORM == 'local':
        base_model_eeg.load_weights('./input/tf-efficientnet-imagenet-weights/efficientnet-b1_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
    if PLATFORM == 'kaggle':
        base_model_eeg.load_weights('/kaggle/input/tf-efficientnet-imagenet-weights/efficientnet-b1_weights_tf_dim_ordering_tf_kernels_autoaugment_notop.h5')
    
    # RESHAPE INPUT 128x256x4 => 512x256x3 MONOTONE IMAGE
    x0_eeg = inp_eeg[:,:,:,:1]
    x1_eeg = inp_eeg[:,:,:,1:2]
    x2_eeg = inp_eeg[:,:,:,2:3]
    x3_eeg = inp_eeg[:,:,:,3:4]
    x_eeg = tf.keras.layers.Concatenate(axis=1)([x0_eeg,x1_eeg,x2_eeg,x3_eeg])
    x_eeg = tf.keras.layers.Reshape((-1, 256, 1))(x_eeg)
    x_eeg = tf.keras.layers.Concatenate(axis=3)([x_eeg,x_eeg,x_eeg])
    
    x_eeg= base_model_eeg(x_eeg)
    x_eeg = tf.keras.layers.GlobalAveragePooling2D()(x_eeg)
    
    
    # CLASS
    
    x = tf.keras.layers.Concatenate(axis=1)([x, x_eeg])
    x = tf.keras.layers.Dense(6,activation='softmax', dtype='float32')(x)
    
    
    # COMPILE MODEL
    model = tf.keras.Model(inputs=[inp, inp_eeg], outputs=x)
    opt = tf.keras.optimizers.Adam(learning_rate = 1e-3)
    # loss = tf.keras.losses.SparseCategoricalCrossentropy()
    loss = tf.keras.losses.CategoricalCrossentropy()

    model.compile(loss=loss, optimizer = opt) 
        
    return model


#%% Train Model
if NEEDTRAIN:
    from sklearn.model_selection import KFold, GroupKFold
    import tensorflow.keras.backend as K, gc
    
    all_oof = []
    all_true = []
    
    gkf = GroupKFold(n_splits=5)
    for i, (train_index, valid_index) in enumerate(gkf.split(train, train.target, train.patient_id)):   
        
        print('#'*25)
        print(f'### Fold {i+1}')
        
        train_gen = DataGenerator(train.iloc[train_index], shuffle=True, batch_size=16, specs = spectrograms, eegs = eegs)
        valid_gen = DataGenerator(train.iloc[valid_index], shuffle=False, batch_size=32, mode='valid', specs = spectrograms, eegs = eegs)
        
        print(f'### train size {len(train_index)}, valid size {len(valid_index)}')
        print('#'*25)
        
        K.clear_session()
        
        callbacks_list = [LR,
                          tf.keras.callbacks.ModelCheckpoint(filepath=f'EB2_v{VER}_f{i}.h5',
                                                              monitor='val_loss', mode='min',
                                                              save_weights_only=True, save_best_only=True),
                          tf.keras.callbacks.EarlyStopping(patience=5, monitor='val_loss', mode='min')]
        
        with strategy.scope():
            model = build_model()
        model.fit(train_gen, verbose=1,
              validation_data = valid_gen,
              epochs=EPOCHS, callbacks = callbacks_list)
            
        oof = model.predict(valid_gen, verbose=1)
        all_oof.append(oof)
        all_true.append(train.iloc[valid_index][TARGETS].values)
        
        del model, oof
        K.clear_session()
        reset_default_graph()
        gc.collect()
        
    all_oof = np.concatenate(all_oof)
    all_true = np.concatenate(all_true)


#%% CV Score for EfficientNetB2
if NEEDTRAIN:
    import sys
    if PLATFORM == 'local':
        sys.path.append('./input/kaggle-kl-div')
    elif PLATFORM == 'kaggle':
        sys.path.append('/kaggle/input/kaggle-kl-div')
    from kaggle_kl_div import score
    
    oof = pd.DataFrame(all_oof.copy())
    oof['id'] = np.arange(len(oof))
    
    true = pd.DataFrame(all_true.copy())
    true['id'] = np.arange(len(true))
    
    cv = score(solution=true, submission=oof, row_id_column_name='id')
    print('CV Score KL-Div for EfficientNetB2 =',cv)


if not NEEDTRAIN:
    #%% Infer Test and Create Submission CSV
    if PLATFORM == 'local':
        test = pd.read_csv('./input/hms-harmful-brain-activity-classification/test.csv')
    elif PLATFORM == 'kaggle':
        test = pd.read_csv('/kaggle/input/hms-harmful-brain-activity-classification/test.csv')
    print('Test shape',test.shape)
    test.head()
    
    # READ ALL SPECTROGRAMS
    if PLATFORM == 'local':
        PATH2 = './input/hms-harmful-brain-activity-classification/test_spectrograms/'
    elif PLATFORM == 'kaggle':
        PATH2 = '/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/'
    
    files2 = os.listdir(PATH2)
    print(f'There are {len(files2)} test spectrogram parquets')
        
    spectrograms2 = {}
    for i,f in enumerate(files2):
        if i%100==0: print(i,', ',end='')
        tmp = pd.read_parquet(f'{PATH2}{f}')
        name = int(f.split('.')[0])
        spectrograms2[name] = tmp.iloc[:,1:].values
        
    # RENAME FOR DATALOADER
    test = test.rename({'spectrogram_id':'spec_id'},axis=1)
    
    
    #%% READ ALL EEGS
    from scipy import signal
    
    # READ ALL SPECTROGRAMS
    if PLATFORM == 'local':
        PATH2 = './input/hms-harmful-brain-activity-classification/test_eegs/'
    elif PLATFORM == 'kaggle':
        PATH2 = '/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/'
        
    files2 = os.listdir(PATH2)
    print(f'There are {len(files2)} test eeg parquets')
      
    eegs2 = {}
    b, a = signal.butter(3, np.float32(filter_range)*2/SFREQ, 'bandpass')
    for i,f in enumerate(files2):
        if i%100==0: print(i,', ',end='')
        raw_eeg = pd.read_parquet(f'{PATH2}{f}')
        name = int(f.split('.')[0])
        
        if len(test[test.eeg_id == name]) > 0:
            time_temp = 0
            time_start =  round(time_temp + (50 - EEG_LENGTH) / 2 * 200) 
            time_stop =  round(time_temp + (50 + EEG_LENGTH) / 2 * 200)
    
            eeg_default = raw_eeg.loc[time_start: (time_stop - 1), :].reset_index(drop=True)
            
            list_eeg = list()
            for region in BRAIN.keys():
    
                eeg = np.zeros((len(BRAIN[region]), eeg_default.shape[0]), dtype=np.float32)
                for chan_i, chan in enumerate(BRAIN[region]):
                    eeg[chan_i, :] = (eeg_default.loc[:, chan.split('-')[0]] - eeg_default.loc[:, chan.split('-')[1]]).values
                
                eeg[np.isnan(eeg)] = 0
                
                if 200 != SFREQ:
                    eeg = signal.resample_poly(eeg, SFREQ, 200, axis=1)
    
                eeg = signal.filtfilt(b, a, eeg, axis=1)
                
                # eeg = (eeg - np.mean(eeg, 1, keepdims=True)) / (np.std(eeg, 1, keepdims=True) + 1e-6)
                
                list_eeg.append(np.reshape(eeg, (eeg.shape[0], eeg.shape[1], 1)))
                
            list_eeg = np.concatenate(list_eeg, 2)
            
            eegs2[name] = list_eeg
    
    
    
    
    # INFER EFFICIENTNET ON TEST
    preds = []
    model = build_model()
    test_gen = DataGenerator(test, shuffle=False, batch_size=32, mode='test', specs = spectrograms2, eegs = eegs2)
    
    for i in range(5):
        print(f'Fold {i+1}')
        model.load_weights(os.path.join(LOAD_MODELS_FROM, f'EB2_v{VER}_f{i}.h5'))
        pred = model.predict(test_gen, verbose=1)
        preds.append(pred)
    pred = np.mean(preds,axis=0)
    print()
    print('Test preds shape',pred.shape)
    
    
    sub = pd.DataFrame({'eeg_id':test.eeg_id.values})
    sub[TARGETS] = pred
    sub.to_csv('submission.csv',index=False)
    print('Submissionn shape',sub.shape)
    sub.head()
    
    
    # SANITY CHECK TO CONFIRM PREDICTIONS SUM TO ONE
    sub.iloc[:,-6:].sum(axis=1)







Looking in links: /kaggle/input/tf-efficientnet-whl-files
Processing /kaggle/input/tf-efficientnet-whl-files/efficientnet-1.1.1-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/tf-efficientnet-whl-files/efficientnet-1.1.1-py3-none-any.whl'



2026-04-28 04:46:02.775380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777351562.788449      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777351562.792492      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 04:46:02.808319: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version = 2.18.0
Using 0 GPU
Mixed precision enabled
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Train non-overlapp eeg_id shape: (15396, 13)


2026-04-28 04:46:10.225860: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ModuleNotFoundError: No module named 'efficientnet'